In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Use the official MediaWiki API to fetch parsed page HTML to avoid 403 Forbidden
api_url = "https://liquipedia.net/pubgmobile/api.php"
params = {
    "action": "parse",
    "page": "PUBG_Mobile_Global_Open/2026/Season_1/Statistics/Teams",
    "format": "json"
}
headers = {
    "User-Agent": "PMGO_Dashboard/1.0 (contact@example.com)"
}

response = requests.get(api_url, params=params, headers=headers)
print(f"API Response Status Code: {response.status_code}")

if response.status_code == 200:
    html = response.json().get("parse", {}).get("text", {}).get("*", "")
    soup = BeautifulSoup(html, "html.parser")
    tables = soup.find_all("table")
    print(f"Found {len(tables)} tables")
else:
    print(f"Failed to fetch data: {response.text}")

API Response Status Code: 200
Found 7 tables


### Bypassing 403 Forbidden via MediaWiki API

Liquipedia uses Cloudflare protection to block automated scraping of HTML pages (returning a `403 Forbidden` error).
However, they provide an official MediaWiki API that permits data access under specific guidelines (using a custom User-Agent and respecting rate limits).
We use the API's `action=parse` action to retrieve the parsed HTML of the team statistics page, allowing us to find the tables programmatically.

In [3]:
import pandas as pd

df = pd.read_csv('../data/team_standings.csv')
print(df.shape)
df.head()

(16, 28)


,Rank,Participant,Total Points,MPe Game,Game 1 P,Game 1 K,Game 2 P,Game 2 K,Game 3 P,Game 3 K,...,Game 8 P,Game 8 K,Game 9 P,Game 9 K,Game 10 P,Game 10 K,Game 11 P,Game 11 K,Game 12 P,Game 12 K
0,1st,4Thrives,122,Game 8,10th,2,1st,9,3rd,8,...,14th,5,8th,1,13th,3,9th,1,3rd,4
1,2nd,ULF Esports,104,Game 11,3rd,5,7th,5,1st,13,...,6th,3,2nd,17,4th,16,5th,2,9th,0
2,3rd,FURIA,97,NaN,1st,16,8th,10,12th,0,...,9th,3,1st,11,7th,0,13th,3,2nd,13
3,4th,S2G Esports,83,NaN,14th,0,15th,0,2nd,3,...,4th,5,16th,0,9th,2,1st,17,6th,7
4,5th,eArena,83,NaN,11th,4,2nd,6,6th,8,...,1st,12,5th,6,8th,0,3rd,5,15th,1


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 28 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Rank          16 non-null     str  
 1   Participant   16 non-null     str  
 2   Total Points  16 non-null     int64
 3   MPe Game      2 non-null      str  
 4   Game 1 P      16 non-null     str  
 5   Game 1 K      16 non-null     int64
 6   Game 2 P      16 non-null     str  
 7   Game 2 K      16 non-null     int64
 8   Game 3 P      16 non-null     str  
 9   Game 3 K      16 non-null     int64
 10  Game 4 P      16 non-null     str  
 11  Game 4 K      16 non-null     int64
 12  Game 5 P      16 non-null     str  
 13  Game 5 K      16 non-null     int64
 14  Game 6 P      16 non-null     str  
 15  Game 6 K      16 non-null     int64
 16  Game 7 P      16 non-null     str  
 17  Game 7 K      16 non-null     int64
 18  Game 8 P      16 non-null     str  
 19  Game 8 K      16 non-null     int64
 20  G

In [7]:
df[['Game 1 P', 'Game 2 P', 'Game 3 P']].head(10)

,Game 1 P,Game 2 P,Game 3 P
0,10th,1st,3rd
1,3rd,7th,1st
2,1st,8th,12th
3,14th,15th,2nd
4,11th,2nd,6th
5,12th,16th,15th
6,5th,6th,5th
7,16th,13th,10th
8,6th,10th,9th
9,9th,5th,7th


In [8]:
df['Rank'].head(10)

0     1st
1     2nd
2     3rd
3     4th
4     5th
5     6th
6     7th
7     8th
8     9th
9    10th
Name: Rank, dtype: str

In [8]:
import re

def clean_rank(value):
    if pd.isna(value):
        return None
    return int(re.sub(r'(st|nd|rd|th)$', '', str(value).strip()))

# Clean the overall Rank column
df['Rank'] = df['Rank'].apply(clean_rank)

# Clean every "Game X P" column
placement_cols = [col for col in df.columns if col.endswith(' P')]
for col in placement_cols:
    df[col] = df[col].apply(clean_rank)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 28 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Rank          16 non-null     int64
 1   Participant   16 non-null     str  
 2   Total Points  16 non-null     int64
 3   MPe Game      2 non-null      str  
 4   Game 1 P      16 non-null     int64
 5   Game 1 K      16 non-null     int64
 6   Game 2 P      16 non-null     int64
 7   Game 2 K      16 non-null     int64
 8   Game 3 P      16 non-null     int64
 9   Game 3 K      16 non-null     int64
 10  Game 4 P      16 non-null     int64
 11  Game 4 K      16 non-null     int64
 12  Game 5 P      16 non-null     int64
 13  Game 5 K      16 non-null     int64
 14  Game 6 P      16 non-null     int64
 15  Game 6 K      16 non-null     int64
 16  Game 7 P      16 non-null     int64
 17  Game 7 K      16 non-null     int64
 18  Game 8 P      16 non-null     int64
 19  Game 8 K      16 non-null     int64
 20  G

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 28 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Rank          16 non-null     int64
 1   Participant   16 non-null     str  
 2   Total Points  16 non-null     int64
 3   MPe Game      2 non-null      str  
 4   Game 1 P      16 non-null     int64
 5   Game 1 K      16 non-null     int64
 6   Game 2 P      16 non-null     int64
 7   Game 2 K      16 non-null     int64
 8   Game 3 P      16 non-null     int64
 9   Game 3 K      16 non-null     int64
 10  Game 4 P      16 non-null     int64
 11  Game 4 K      16 non-null     int64
 12  Game 5 P      16 non-null     int64
 13  Game 5 K      16 non-null     int64
 14  Game 6 P      16 non-null     int64
 15  Game 6 K      16 non-null     int64
 16  Game 7 P      16 non-null     int64
 17  Game 7 K      16 non-null     int64
 18  Game 8 P      16 non-null     int64
 19  Game 8 K      16 non-null     int64
 20  G

In [9]:
kill_cols = [col for col in df.columns if col.endswith(' K')]
df['Total Kill Points'] = df[kill_cols].sum(axis=1)
df['Total Placement Points'] = df['Total Points'] - df['Total Kill Points']

df[['Participant', 'Total Points', 'Total Kill Points', 'Total Placement Points']].sort_values('Total Points', ascending=False)

,Participant,Total Points,Total Kill Points,Total Placement Points
0,4Thrives,122,77,45
1,ULF Esports,104,70,34
2,FURIA,97,65,32
3,S2G Esports,83,53,30
4,eArena,83,56,27
5,TT Project,77,51,26
6,GOAT Team,75,49,26
7,Aurora,73,47,26
8,BOOM Esports,71,49,22
9,Team Flash,69,45,24


In [11]:
df.to_csv('../data/team_standings_clean.csv', index=False)
print("Saved!")

Saved!


In [10]:
game_rows = []

for _, row in df.iterrows():
    for game_num in range(1, 13):
        p_col = f"Game {game_num} P"
        k_col = f"Game {game_num} K"
        game_rows.append({
            "Participant": row["Participant"],
            "Game": game_num,
            "Placement": row[p_col],
            "Kill Points": row[k_col],
        })

games_df = pd.DataFrame(game_rows)
games_df.head(15)

,Participant,Game,Placement,Kill Points
0,4Thrives,1,10,2
1,4Thrives,2,1,9
2,4Thrives,3,3,8
3,4Thrives,4,16,0
4,4Thrives,5,1,16
5,4Thrives,6,1,12
6,4Thrives,7,4,16
7,4Thrives,8,14,5
8,4Thrives,9,8,1
9,4Thrives,10,13,3


In [12]:
placement_points_map = {
    1: 10, 2: 6, 3: 5, 4: 4, 5: 3, 6: 2,
    7: 1, 8: 1, 9: 0, 10: 0, 11: 0, 12: 0,
    13: 0, 14: 0, 15: 0, 16: 0
}

games_df["Placement Points"] = games_df["Placement"].map(placement_points_map)
games_df["Game Total Points"] = games_df["Placement Points"] + games_df["Kill Points"]
games_df.head(15)

,Participant,Game,Placement,Kill Points,Placement Points,Game Total Points
0,4Thrives,1,10,2,0,2
1,4Thrives,2,1,9,10,19
2,4Thrives,3,3,8,5,13
3,4Thrives,4,16,0,0,0
4,4Thrives,5,1,16,10,26
5,4Thrives,6,1,12,10,22
6,4Thrives,7,4,16,4,20
7,4Thrives,8,14,5,0,5
8,4Thrives,9,8,1,1,2
9,4Thrives,10,13,3,0,3


In [13]:
games_df = games_df.sort_values(["Participant", "Game"])
games_df["Cumulative Points"] = games_df.groupby("Participant")["Game Total Points"].cumsum()
games_df.head(15)

,Participant,Game,Placement,Kill Points,Placement Points,Game Total Points,Cumulative Points
0,4Thrives,1,10,2,0,2,2
1,4Thrives,2,1,9,10,19,21
2,4Thrives,3,3,8,5,13,34
3,4Thrives,4,16,0,0,0,34
4,4Thrives,5,1,16,10,26,60
5,4Thrives,6,1,12,10,22,82
6,4Thrives,7,4,16,4,20,102
7,4Thrives,8,14,5,0,5,107
8,4Thrives,9,8,1,1,2,109
9,4Thrives,10,13,3,0,3,112


In [14]:
check = games_df.groupby("Participant")["Game Total Points"].sum().reset_index()
check = check.merge(df[["Participant", "Total Points"]], on="Participant")
check["Difference"] = check["Total Points"] - check["Game Total Points"]
check.sort_values("Difference", ascending=False)

,Participant,Game Total Points,Total Points,Difference
0,4Thrives,122,122,0
1,721 Esports,51,51,0
2,Aurora,73,73,0
3,BOOM Esports,71,71,0
4,Bigetron,53,53,0
5,FURIA,97,97,0
6,GOAT Team,75,75,0
7,Gaming Stars,59,59,0
8,Geekay,54,54,0
9,Horaa Esports,59,59,0


In [16]:
games_df.to_csv('../data/games_long_clean.csv', index=False)
print("Saved!")

Saved!


In [18]:
players_df = pd.read_csv('../data/player_stats.csv')
print(players_df.shape)
players_df.head(10)

(64, 6)


,Player,Elims,Avg. Dmg.\nDealt,Assists,Knocks,KD\nRatio
0,Kecth,27,420,13,19,2.25
1,Ayala,22,371,12,21,1.83
2,HamsiG,22,352,7,20,1.83
3,NEOZ,21,321,7,21,1.75
4,Silenceee,20,335,15,21,1.67
5,T24OP,20,329,13,17,1.67
6,SkY,19,373,10,20,1.58
7,IQ,19,349,12,21,1.58
8,Focus,19,333,6,19,1.58
9,HUZAIFA,19,300,6,17,1.58


In [20]:
players_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Player           64 non-null     str    
 1   Elims            64 non-null     int64  
 2   Avg. Dmg.
Dealt  64 non-null     int64  
 3   Assists          64 non-null     int64  
 4   Knocks           64 non-null     int64  
 5   KD
Ratio         64 non-null     float64
dtypes: float64(1), int64(4), str(1)
memory usage: 3.5 KB


In [22]:
players_df = pd.read_csv('../data/player_stats.csv')
players_df.info()
players_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Player           64 non-null     str    
 1   Team             64 non-null     str    
 2   Elims            64 non-null     int64  
 3   Avg. Dmg.
Dealt  64 non-null     int64  
 4   Assists          64 non-null     int64  
 5   Knocks           64 non-null     int64  
 6   KD
Ratio         64 non-null     float64
dtypes: float64(1), int64(4), str(2)
memory usage: 4.7 KB


,Player,Team,Elims,Avg. Dmg.\nDealt,Assists,Knocks,KD\nRatio
0,Kecth,ULF Esports,27,420,13,19,2.25
1,Ayala,FURIA,22,371,12,21,1.83
2,HamsiG,S2G Esports,22,352,7,20,1.83
3,NEOZ,TT Project,21,321,7,21,1.75
4,Silenceee,FURIA,20,335,15,21,1.67


In [24]:
players_df.columns = players_df.columns.str.replace('\n', ' ', regex=False)
players_df.columns.tolist()

['Player', 'Team', 'Elims', 'Avg. Dmg. Dealt', 'Assists', 'Knocks', 'KD Ratio']

In [26]:
players_df.to_csv('../data/player_stats_clean.csv', index=False)
print("Saved!")

Saved!


In [28]:
erangel_df = pd.read_csv('../data/map_erangel.csv')
print(erangel_df.shape)
erangel_df.head()

(17, 22)


,#,Participant,Total\nPoints,Matches\nPlayed,Points,Unnamed: 5,Averages,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Points Earned Summary,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21
0,NaN,NaN,NaN,NaN,Place,Elims,Placement,Place Pts.,Elims,Total Pts.,...,NaN,Top 5,Top 8,> 8th,0,1 ~ 5,6 ~ 10,11 ~ 15,16 ~ 20,20+
1,1.0,ULF Esports,77.0,6.0,23,54,5.83,3.83,9,12.83,...,-,3,5,1,1,1,1,-,1,2
2,2.0,eArena,59.0,6.0,22,37,5.5,3.67,6.17,9.83,...,-,3,5,1,-,2,2,1,-,1
3,3.0,4Thrives Esports,42.0,6.0,16,26,9.17,2.67,4.33,7,...,1,2,3,3,1,3,-,1,1,-
4,4.0,Geekay Esports,40.0,6.0,21,19,6.67,3.5,3.17,6.67,...,2,3,4,2,2,1,2,-,1,-


In [30]:
# Reload, skipping the messy header rows, and manually assign clean column names
erangel_df = pd.read_csv('../data/map_erangel.csv', skiprows=2, header=None)

erangel_df = erangel_df.iloc[:, [1, 2, 3, 6, 8]]  # Participant, Total Points, Matches Played, Avg Placement, Avg Elims
erangel_df.columns = ['Participant', 'Total Points', 'Matches Played', 'Avg Placement', 'Avg Elims']
erangel_df['Map'] = 'Erangel'

erangel_df.head()

,Participant,Total Points,Matches Played,Avg Placement,Avg Elims,Map
0,ULF Esports,77,6,5.83,9.00,Erangel
1,eArena,59,6,5.50,6.17,Erangel
2,4Thrives Esports,42,6,9.17,4.33,Erangel
3,Geekay Esports,40,6,6.67,3.17,Erangel
4,FURIA,38,6,8.17,4.33,Erangel


In [32]:
rondo_df = pd.read_csv('../data/map_rondo.csv', skiprows=2, header=None)
rondo_df = rondo_df.iloc[:, [1, 2, 3, 6, 8]]
rondo_df.columns = ['Participant', 'Total Points', 'Matches Played', 'Avg Placement', 'Avg Elims']
rondo_df['Map'] = 'Rondo'
rondo_df.head()

,Participant,Total Points,Matches Played,Avg Placement,Avg Elims,Map
0,FURIA,28,2,7.0,9.0,Rondo
1,Horaa Esports,28,2,9.0,11.0,Rondo
2,BOOM Esports,27,2,4.5,10.0,Rondo
3,TT Project,25,2,6.5,7.5,Rondo
4,4Thrives Esports,22,2,7.0,9.0,Rondo


In [34]:
miramar_df = pd.read_csv('../data/map_miramar.csv', skiprows=2, header=None)
miramar_df = miramar_df.iloc[:, [1, 2, 3, 6, 8]]
miramar_df.columns = ['Participant', 'Total Points', 'Matches Played', 'Avg Placement', 'Avg Elims']
miramar_df['Map'] = 'Miramar'
miramar_df.head()

,Participant,Total Points,Matches Played,Avg Placement,Avg Elims,Map
0,4Thrives Esports,58,4,3.50,8.25,Miramar
1,Aurora Gaming,45,4,4.75,6.50,Miramar
2,S2G Esports,45,4,5.75,7.75,Miramar
3,GOAT Team,33,4,7.00,5.75,Miramar
4,FURIA,31,4,7.25,5.25,Miramar


In [35]:
all_maps_df = pd.concat([erangel_df, rondo_df, miramar_df], ignore_index=True)
all_maps_df.to_csv('../data/map_performance_clean.csv', index=False)
print(all_maps_df.shape)
all_maps_df.head(20)

(48, 6)


,Participant,Total Points,Matches Played,Avg Placement,Avg Elims,Map
0,ULF Esports,77,6,5.83,9.00,Erangel
1,eArena,59,6,5.50,6.17,Erangel
2,4Thrives Esports,42,6,9.17,4.33,Erangel
3,Geekay Esports,40,6,6.67,3.17,Erangel
4,FURIA,38,6,8.17,4.33,Erangel
5,Bigetron by Vitality,36,6,8.17,4.00,Erangel
6,TT Project,36,6,9.00,4.00,Erangel
7,GOAT Team,34,6,6.83,3.50,Erangel
8,Team Flash,33,6,7.17,3.17,Erangel
9,721 Esports,28,6,10.33,2.83,Erangel


In [37]:
team_adv_df = pd.read_csv('../data/team_advanced_stats.csv')
team_adv_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Grand Finals Statistics  17 non-null     str  
 1   Unnamed: 1               17 non-null     str  
 2   Unnamed: 2               17 non-null     str  
 3   Unnamed: 3               17 non-null     str  
 4   Unnamed: 4               17 non-null     str  
 5   Unnamed: 5               17 non-null     str  
 6   Unnamed: 6               17 non-null     str  
 7   Unnamed: 7               17 non-null     str  
 8   Unnamed: 8               17 non-null     str  
 9   Unnamed: 9               17 non-null     str  
 10  Unnamed: 10              17 non-null     str  
 11  Unnamed: 11              17 non-null     str  
 12  Unnamed: 12              17 non-null     str  
 13  Unnamed: 13              17 non-null     str  
 14  Unnamed: 14              17 non-null     str  
 15  Unnamed: 15        

In [39]:
player_adv_df = pd.read_csv('../data/player_advanced_stats.csv')
player_adv_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 65 entries, 0 to 64
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Grand Finals Statistics  65 non-null     str  
 1   Unnamed: 1               65 non-null     str  
 2   Unnamed: 2               65 non-null     str  
 3   Unnamed: 3               65 non-null     str  
 4   Unnamed: 4               65 non-null     str  
 5   Unnamed: 5               65 non-null     str  
 6   Unnamed: 6               65 non-null     str  
 7   Unnamed: 7               65 non-null     str  
 8   Unnamed: 8               65 non-null     str  
 9   Unnamed: 9               65 non-null     str  
 10  Unnamed: 10              65 non-null     str  
 11  Unnamed: 11              65 non-null     str  
 12  Unnamed: 12              65 non-null     str  
 13  Unnamed: 13              65 non-null     str  
 14  Unnamed: 14              65 non-null     str  
 15  Unnamed: 15        

In [41]:
team_adv_df = pd.read_csv('../data/team_advanced_stats.csv', skiprows=2, header=None)

team_adv_df.columns = [
    'Rank', 'Team', 'Total Points', 'Survival Time', 'Damage Received',
    'Healing Done', 'Avg Dmg Dealt', 'Damage Dealt', 'Assists', 'Knocks',
    'Headshots', 'Max Elim Range', 'Smokes Used', 'Grenades Used',
    'Grenade Elims', 'Teammates Rescued', 'Airdrops Taken',
    'Distance Driven', 'Distance Walked', 'Distance Traveled'
]

# Strip commas and convert numeric columns
numeric_cols = team_adv_df.columns.difference(['Team'])
for col in numeric_cols:
    team_adv_df[col] = team_adv_df[col].astype(str).str.replace(',', '').astype(float)

team_adv_df.info()
team_adv_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Rank               16 non-null     float64
 1   Team               16 non-null     str    
 2   Total Points       16 non-null     float64
 3   Survival Time      16 non-null     float64
 4   Damage Received    16 non-null     float64
 5   Healing Done       16 non-null     float64
 6   Avg Dmg Dealt      16 non-null     float64
 7   Damage Dealt       16 non-null     float64
 8   Assists            16 non-null     float64
 9   Knocks             16 non-null     float64
 10  Headshots          16 non-null     float64
 11  Max Elim Range     16 non-null     float64
 12  Smokes Used        16 non-null     float64
 13  Grenades Used      16 non-null     float64
 14  Grenade Elims      16 non-null     float64
 15  Teammates Rescued  16 non-null     float64
 16  Airdrops Taken     16 non-null     floa

,Rank,Team,Total Points,Survival Time,Damage Received,Healing Done,Avg Dmg Dealt,Damage Dealt,Assists,Knocks,Headshots,Max Elim Range,Smokes Used,Grenades Used,Grenade Elims,Teammates Rescued,Airdrops Taken,Distance Driven,Distance Walked,Distance Traveled
0,1.0,4Thrives Esports,122.0,57120.0,12819.0,5365.0,1276.0,15315.0,41.0,71.0,16.0,321.0,162.0,90.0,14.0,17.0,0.0,288996.0,76310.0,365306.0
1,2.0,ULF Esports,104.0,60351.0,12109.0,5662.0,1255.0,15071.0,51.0,58.0,16.0,300.0,109.0,99.0,12.0,13.0,3.0,339387.0,75024.0,414411.0
2,3.0,FURIA,97.0,60567.0,14307.0,9428.0,1193.0,14327.0,53.0,67.0,11.0,365.0,141.0,84.0,8.0,22.0,4.0,362699.0,79926.0,442625.0
3,4.0,S2G Esports,83.0,58415.0,12016.0,7865.0,982.0,11792.0,27.0,50.0,7.0,356.0,110.0,73.0,2.0,16.0,4.0,392030.0,72844.0,464874.0
4,5.0,eArena,83.0,57900.0,10872.0,6208.0,912.0,10952.0,52.0,48.0,11.0,293.0,167.0,72.0,10.0,12.0,1.0,343100.0,79181.0,422281.0


In [43]:
player_adv_df = pd.read_csv('../data/player_advanced_stats.csv', skiprows=2, header=None)

player_adv_df.columns = [
    'Rank', 'Player', 'Team', 'Matches Played', 'Elims', 'Avg Dmg Dealt',
    'Damage Dealt', 'Assists', 'Knocks', 'Headshots', 'KD Ratio',
    'Max Elim Range', 'Survival Time', 'Damage Received', 'Healing Done',
    'Deaths', 'Smokes Used', 'Grenades Used', 'Grenade Elims',
    'Airdrops Taken', 'Distance Driven', 'Distance Walked', 'Distance Traveled'
]

numeric_cols = player_adv_df.columns.difference(['Player', 'Team'])
for col in numeric_cols:
    player_adv_df[col] = player_adv_df[col].astype(str).str.replace(',', '').astype(float)

player_adv_df.info()
player_adv_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Rank               64 non-null     float64
 1   Player             64 non-null     str    
 2   Team               64 non-null     str    
 3   Matches Played     64 non-null     float64
 4   Elims              64 non-null     float64
 5   Avg Dmg Dealt      64 non-null     float64
 6   Damage Dealt       64 non-null     float64
 7   Assists            64 non-null     float64
 8   Knocks             64 non-null     float64
 9   Headshots          64 non-null     float64
 10  KD Ratio           64 non-null     float64
 11  Max Elim Range     64 non-null     float64
 12  Survival Time      64 non-null     float64
 13  Damage Received    64 non-null     float64
 14  Healing Done       64 non-null     float64
 15  Deaths             64 non-null     float64
 16  Smokes Used        64 non-null     floa

,Rank,Player,Team,Matches Played,Elims,Avg Dmg Dealt,Damage Dealt,Assists,Knocks,Headshots,...,Damage Received,Healing Done,Deaths,Smokes Used,Grenades Used,Grenade Elims,Airdrops Taken,Distance Driven,Distance Walked,Distance Traveled
0,1.0,Kecth,ULF Esports,12.0,27.0,420.0,5043.0,13.0,19.0,4.0,...,2924.0,1719.0,11.0,23.0,28.0,5.0,1.0,90013.0,20044.0,110057.0
1,2.0,Ayala,FURIA,12.0,22.0,371.0,4463.0,12.0,21.0,3.0,...,3930.0,2696.0,10.0,50.0,33.0,3.0,1.0,93375.0,20763.0,114138.0
2,3.0,HamsiG,S2G Esports,12.0,22.0,352.0,4232.0,7.0,20.0,3.0,...,2955.0,1951.0,11.0,25.0,26.0,2.0,2.0,94693.0,19319.0,114012.0
3,4.0,NEOZ,TT Project,12.0,21.0,321.0,3859.0,7.0,21.0,3.0,...,2508.0,1131.0,11.0,18.0,11.0,3.0,0.0,80754.0,16393.0,97147.0
4,5.0,Silenceee,FURIA,12.0,20.0,335.0,4030.0,15.0,21.0,5.0,...,3721.0,2770.0,10.0,25.0,22.0,3.0,1.0,78127.0,22704.0,100831.0


In [45]:
team_adv_df.to_csv('../data/team_advanced_stats_clean.csv', index=False)
player_adv_df.to_csv('../data/player_advanced_stats_clean.csv', index=False)
print("Saved both!")

Saved both!


In [49]:
def load_stage_standings(filepath, stage_name):
    df = pd.read_csv(filepath, skiprows=2, header=None)
    df.columns = ['Rank', 'Team', 'Matches Played', 'Place Points', 'Elim Points', 'Chicken Dinners', 'Total Points']
    numeric_cols = ['Rank', 'Matches Played', 'Place Points', 'Elim Points', 'Chicken Dinners', 'Total Points']
    for col in numeric_cols:
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)
    df['Stage'] = stage_name
    return df

group_a_teams = load_stage_standings('../data/group_stage_a_teams.csv', 'Group A')
group_b_teams = load_stage_standings('../data/group_stage_b_teams.csv', 'Group B')
survival_teams = load_stage_standings('../data/survival_stage_teams.csv', 'Survival Stage')

group_a_teams.head()


,Rank,Team,Matches Played,Place Points,Elim Points,Chicken Dinners,Total Points,Stage
0,1.0,Horaa Esports,6.0,22.0,53.0,1.0,75.0,Group A
1,2.0,eArena,6.0,24.0,24.0,2.0,48.0,Group A
2,3.0,Team Pandum,6.0,17.0,29.0,1.0,46.0,Group A
3,4.0,S2G Esports,6.0,20.0,24.0,2.0,44.0,Group A
4,5.0,Aurora Gaming,6.0,13.0,29.0,0.0,42.0,Group A


In [51]:
group_b_teams.head()


,Rank,Team,Matches Played,Place Points,Elim Points,Chicken Dinners,Total Points,Stage
0,1.0,Bigetron by Vitality,6.0,30.0,52.0,1.0,82.0,Group B
1,2.0,ULF Esports,6.0,22.0,40.0,1.0,62.0,Group B
2,3.0,Gaming Stars Esports,6.0,21.0,39.0,1.0,60.0,Group B
3,4.0,TT Project,6.0,9.0,39.0,0.0,48.0,Group B
4,5.0,Geekay Esports,6.0,22.0,24.0,1.0,46.0,Group B


In [53]:
survival_teams.head()

,Rank,Team,Matches Played,Place Points,Elim Points,Chicken Dinners,Total Points,Stage
0,1.0,Team Flash,6.0,28.0,41.0,2.0,69.0,Survival Stage
1,2.0,GOAT Team,6.0,21.0,39.0,1.0,60.0,Survival Stage
2,3.0,4Thrives Esports,6.0,15.0,42.0,1.0,57.0,Survival Stage
3,4.0,721 Esports,6.0,20.0,25.0,0.0,45.0,Survival Stage
4,5.0,AlUla Club Esports,6.0,12.0,32.0,0.0,44.0,Survival Stage


In [54]:
all_stages_df = pd.concat([group_a_teams, group_b_teams, survival_teams], ignore_index=True)
all_stages_df.to_csv('../data/stage_standings_clean.csv', index=False)
print(all_stages_df.shape)
all_stages_df.head(20)

(48, 8)


,Rank,Team,Matches Played,Place Points,Elim Points,Chicken Dinners,Total Points,Stage
0,1.0,Horaa Esports,6.0,22.0,53.0,1.0,75.0,Group A
1,2.0,eArena,6.0,24.0,24.0,2.0,48.0,Group A
2,3.0,Team Pandum,6.0,17.0,29.0,1.0,46.0,Group A
3,4.0,S2G Esports,6.0,20.0,24.0,2.0,44.0,Group A
4,5.0,Aurora Gaming,6.0,13.0,29.0,0.0,42.0,Group A
5,6.0,FURIA,6.0,10.0,31.0,0.0,41.0,Group A
6,7.0,Boars Gaming,6.0,13.0,27.0,0.0,40.0,Group A
7,8.0,IDA Esports,6.0,8.0,31.0,0.0,39.0,Group A
8,9.0,VOIN Esports,6.0,18.0,17.0,0.0,35.0,Group A
9,10.0,4Thrives Esports,6.0,8.0,24.0,0.0,32.0,Group A
